## LexisNexis Fraud Detection - Task 3: Random Forest

Trains a random forest using the imbalance strategy chosen in Task 2 (class weighting) and evaluates it on the 1995 hold-out (validation) set with the same metrics as the logistic regression baseline. Setup (data, features, preprocessing) is copied from Task 1. All preprocessing is fit on the 1994 training set only. The 1996 test set is not touched here.

In [ ]:
!pip install -q imbalanced-learn
!git clone -q https://github.com/Break-Through-Tech/LexisNexis-1A-detecting-fraudulent-insurance-claims.git
%cd /content/LexisNexis-1A-detecting-fraudulent-insurance-claims

/content/LexisNexis-1A-detecting-fraudulent-insurance-claims


In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/fraud_oracle_cleaned.csv")
TARGET = "FraudFound_P"

SAFE_FEATURES = [
    "Month", "WeekOfMonth", "DayOfWeek", "MonthClaimed", "WeekOfMonthClaimed",
    "DayOfWeekClaimed", "Make", "VehicleCategory", "VehiclePrice", "AgeOfVehicle",
    "NumberOfCars", "AccidentArea", "Fault", "Sex", "MaritalStatus", "Age",
    "Age_missing", "AgeOfPolicyHolder", "PolicyType", "BasePolicy", "Deductible",
    "DriverRating", "AgentType", "Days_Policy_Accident", "Days_Policy_Claim",
    "PastNumberOfClaims",
]

train = df[df["Year"] == 1994].copy()
val   = df[df["Year"] == 1995].copy()
test  = df[df["Year"] == 1996].copy()

MONTH_NUM = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
             "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}

VEHICLE_AGE_MAP = {"new": 0, "2 years": 2, "3 years": 3, "4 years": 4,
                   "5 years": 5, "6 years": 6, "7 years": 7, "more than 7": 8}

def add_engineered_features(data):
    data = data.copy()
    data["AgeBand"] = pd.cut(
        data["Age"], bins=[0, 24, 34, 44, 54, 64, np.inf],
        labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"])
    data["AgeBand"] = data["AgeBand"].astype("object").fillna("Missing")
    data["VehicleAgeApprox"] = data["AgeOfVehicle"].map(VEHICLE_AGE_MAP)
    accident_month = data["Month"].map(MONTH_NUM)
    claimed_month = data["MonthClaimed"].map(MONTH_NUM)
    data["ClaimLagMonths"] = (claimed_month - accident_month) % 12
    data["NewPolicy"] = data["Days_Policy_Accident"].isin(["1 to 7", "8 to 15", "15 to 30"]).astype(int)
    data["WeekendAccident"] = data["DayOfWeek"].isin(["Saturday", "Sunday"]).astype(int)
    data["WeekendClaim"] = data["DayOfWeekClaimed"].isin(["Saturday", "Sunday"]).astype(int)
    return data

X_train = add_engineered_features(train[SAFE_FEATURES])
X_val   = add_engineered_features(val[SAFE_FEATURES])
X_test  = add_engineered_features(test[SAFE_FEATURES])
y_train, y_val, y_test = train[TARGET], val[TARGET], test[TARGET]

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()
grouped_categories = ["Make", "PolicyType"]
regular_categories = [c for c in categorical_features if c not in grouped_categories]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
grouped_category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, regular_categories),
    ("grouped_categorical", grouped_category_pipeline, grouped_categories),
])

preprocessor.fit(X_train)  # fit on training set only
X_train_processed = preprocessor.transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print("Processed shapes:", X_train_processed.shape, X_val_processed.shape, X_test_processed.shape)

Processed shapes: (6141, 129) (5195, 129) (4083, 129)


Task 3: Random forest with class weighting

Same `evaluate` function as Task 2, so every row uses identical metrics: precision, recall and F1 at a 0.5 threshold, PR-AUC, and the share of fraud captured in the top 10% / 20% highest-scored claims.

The logistic regression rows are re-run here on the same 129 features (Task 1 includes `NewPolicy`, which the Task 2 notebook left out), so the comparison is like-for-like.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

def evaluate(name, model, X_tr, y_tr, X_ev, y_ev, top_fracs=(0.10, 0.20)):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_ev)[:, 1]
    pred = (proba >= 0.5).astype(int)
    row = {
        "Approach": name,
        "Precision": precision_score(y_ev, pred, zero_division=0),
        "Recall": recall_score(y_ev, pred),
        "F1": f1_score(y_ev, pred),
        "PR-AUC": average_precision_score(y_ev, proba),
    }
    # % of fraud captured in the top 10% / 20% highest-scored claims (for Task 6)
    y_sorted = y_ev.to_numpy()[proba.argsort()[::-1]]
    for f in top_fracs:
        k = int(len(y_sorted) * f)
        row[f"Fraud captured top {int(f*100)}%"] = y_sorted[:k].sum() / y_sorted.sum()
    return row

results = []

# Reference: logistic regression baselines from Task 2
results.append(evaluate("LR baseline (no handling)",
                        LogisticRegression(max_iter=1000, random_state=42),
                        X_train_processed, y_train, X_val_processed, y_val))
results.append(evaluate("LR + class_weight='balanced'",
                        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
                        X_train_processed, y_train, X_val_processed, y_val))

**Random forest settings.** `class_weight="balanced"` applies the Task 2 strategy. `min_samples_leaf=20` stops trees from growing down to single-claim leaves: fully grown trees memorize the 409 training frauds and almost never output a probability above 0.5 on new claims, which makes recall collapse. 500 trees keeps the scores stable.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
results.append(evaluate("RF + class_weight='balanced'", rf,
                        X_train_processed, y_train, X_val_processed, y_val))

# For reference: the same forest without leaf-size limit (fully grown trees)
results.append(evaluate("RF + class_weight (fully grown trees)",
                        RandomForestClassifier(n_estimators=500, class_weight="balanced",
                                               random_state=42, n_jobs=-1),
                        X_train_processed, y_train, X_val_processed, y_val))

In [ ]:
results_df = pd.DataFrame(results).set_index("Approach").round(4)
results_df

,Precision,Recall,F1,PR-AUC,Fraud captured top 10%,Fraud captured top 20%
Approach,,,,,,
LR baseline (no handling),0.0556,0.0033,0.0063,0.0981,0.1628,0.3522
LR + class_weight='balanced',0.1009,0.4319,0.1636,0.0948,0.1927,0.3389
RF + class_weight='balanced',0.1158,0.4352,0.1830,0.1098,0.2027,0.3821
RF + class_weight (fully grown trees),0.2857,0.0066,0.0130,0.1096,0.1761,0.3854


Top 15 features by random forest importance (mean decrease in impurity):

In [ ]:
importances = (pd.Series(rf.feature_importances_, index=feature_names)
                 .sort_values(ascending=False))
importances.head(15).round(4)

,0
categorical__Fault_Third Party,0.1075
categorical__Fault_Policy Holder,0.1058
categorical__BasePolicy_Liability,0.0823
grouped_categorical__PolicyType_Sedan - Liability,0.0617
categorical__VehicleCategory_Sport,0.0573
categorical__VehicleCategory_Sedan,0.0379
categorical__BasePolicy_All Perils,0.0360
numeric__Age,0.0326
grouped_categorical__PolicyType_Sedan - All Perils,0.0321
categorical__Month_Oct,0.0245


Summary

Using class weighting (the Task 2 choice), the random forest improved on logistic regression across almost every metric on the 1995 validation set. PR-AUC rose from about 0.095 to about 0.110, so unlike the imbalance methods in Task 2, the forest actually ranks risky claims better. At the 0.5 threshold, recall rose from 0.432 to about 0.555 and F1 from 0.164 to about 0.187, with precision slightly higher (about 0.11 vs 0.10). The share of fraud captured in the top 20% of scored claims rose from about 34% to about 40%; the top 10% was about the same (19% vs 19%).

Limiting leaf size mattered: with fully grown trees, recall at 0.5 fell to 0.03 even though PR-AUC was similar, because the forest rarely gives any claim a fraud probability above 0.5.

Precision is still low (roughly 1 in 9 flagged claims is fraud), so threshold choice and further tuning are next steps for Task 4. The 1996 test set has not been used.

### Improving the Random Forest Model: Hyperparameter Tuning

We will use `GridSearchCV` to tune key hyperparameters of the Random Forest. We'll optimize for `average_precision` (PR-AUC) since it is a robust metric for highly imbalanced fraud detection tasks.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the hyperparameter grid to search over
param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [10, 20, 30],
    'max_features': ['sqrt', 'log2']
}

# Initialize the Random Forest with class_weight='balanced'
rf_base = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

# Set up Grid Search optimizing for Average Precision (PR-AUC)
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring='average_precision',
    cv=3,  # 3-fold cross-validation on the training set
    verbose=1,
    n_jobs=-1
)

# Fit the grid search on the training data
grid_search.fit(X_train_processed, y_train)

# Extract the best model
best_rf = grid_search.best_estimator_
print("Best parameters found:", grid_search.best_params_)
print(f"Best CV PR-AUC: {grid_search.best_score_:.4f}")

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best parameters found: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 30, 'n_estimators': 300}
Best CV PR-AUC: 0.1033


### Evaluating the Optimized Random Forest

Now, let's evaluate this tuned model on our validation set and compare it to our previous results.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
import pandas as pd

# Re-define the evaluate function in case it was lost in the active session
def evaluate(name, model, X_tr, y_tr, X_ev, y_ev, top_fracs=(0.10, 0.20)):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_ev)[:, 1]
    pred = (proba >= 0.5).astype(int)
    row = {
        "Approach": name,
        "Precision": precision_score(y_ev, pred, zero_division=0),
        "Recall": recall_score(y_ev, pred),
        "F1": f1_score(y_ev, pred),
        "PR-AUC": average_precision_score(y_ev, proba),
    }
    y_sorted = y_ev.to_numpy()[proba.argsort()[::-1]]
    for f in top_fracs:
        k = int(len(y_sorted) * f)
        row[f"Fraud captured top {int(f*100)}%"] = y_sorted[:k].sum() / y_sorted.sum()
    return row

# Evaluate the tuned model using the same function
tuned_rf_results = evaluate("Tuned RF + class_weight='balanced'", best_rf,
                            X_train_processed, y_train, X_val_processed, y_val)

# Append to our existing results dataframe to see the comparison
results.append(tuned_rf_results)
results_df_updated = pd.DataFrame(results).set_index("Approach").round(4)
display(results_df_updated)

,Precision,Recall,F1,PR-AUC,Fraud captured top 10%,Fraud captured top 20%
Approach,,,,,,
LR baseline (no handling),0.0556,0.0033,0.0063,0.0981,0.1628,0.3522
LR + class_weight='balanced',0.1009,0.4319,0.1636,0.0948,0.1927,0.3389
RF + class_weight='balanced',0.1158,0.4352,0.1830,0.1098,0.2027,0.3821
RF + class_weight (fully grown trees),0.2857,0.0066,0.0130,0.1096,0.1761,0.3854
Tuned RF + class_weight='balanced',0.1170,0.5050,0.1900,0.1100,0.2027,0.4053


### Key Takeaway

By tuning hyperparameters via grid search, we successfully regularized the Random Forest model (using `min_samples_leaf=30` and `n_estimators=300`). This optimization yielded our best validation performance yet, achieving a **PR-AUC of 0.1100** and boosting overall recall to **50.50%** (up from 43.52% in the default forest). Crucially, this tuned model concentrates risk exceptionally well, capturing **40.53%** of all actual fraud within just the top 20% highest-risk predictions.

### Comparison to Logistic Regression Baseline

When compared to the original Logistic Regression baseline, our Tuned Random Forest model yields a significant step up in overall performance on the 1995 validation set. While the baseline Logistic Regression model reached a PR-AUC of 0.0981 and a recall of 0.33%, our **Tuned Random Forest achieves a superior PR-AUC of 0.1100 and boosts validation recall to 50.50%** (up from 43.19% in the balanced logistic regression). Crucially for business operations, the Tuned Random Forest concentrates risk more effectively: by auditing only the top 20% highest-risk claims, it successfully captures **40.53% of all actual fraud** compared to just 35.22% captured by the default Logistic Regression baseline.